# CAMB, Capse.jl, and jaxcapse

A minimal TT comparison at one point in the trained domain. The same nine parameters go to both emulators and the CAMB 2.0.4 + CosmoRec reference worker used by this dataset. Capse and Python predictions are lensed $D_\ell$ in $\mu K^2$.

Run a Julia kernel from the Capse.jl checkout root (or `notebooks/`) using `notebooks/Project.toml`. On this machine the first cell finds the jaxcapse Poetry Python automatically. Elsewhere set `JAXCAPSE_PYTHON`. `CAMB_COSMOREC_ROOT` can override the neighboring CAMB source path. The sibling `jaxcapse` and `emulator-zoo` checkouts provide Python and the canonical CAMB worker.

In [1]:
using Pkg

notebooks_dir = isfile(joinpath(pwd(), "notebooks", "Project.toml")) ?
    joinpath(pwd(), "notebooks") : pwd()
@assert isfile(joinpath(notebooks_dir, "Project.toml")) "Start Jupyter from the Capse.jl checkout root or notebooks/"
Pkg.activate(notebooks_dir)

workspace = dirname(dirname(notebooks_dir))
jaxcapse_root = get(ENV, "JAXCAPSE_ROOT", joinpath(workspace, "jaxcapse"))
poetry_envs = joinpath(homedir(), ".cache", "pypoetry", "virtualenvs")
jaxcapse_python = get(ENV, "JAXCAPSE_PYTHON", "")
if isempty(jaxcapse_python)
    candidates = isdir(poetry_envs) ? filter(name -> startswith(name, "jaxcapse-"), readdir(poetry_envs)) : String[]
    length(candidates) == 1 || error("Set JAXCAPSE_PYTHON to the jaxcapse Poetry interpreter.")
    jaxcapse_python = joinpath(poetry_envs, only(candidates), "bin", "python")
end
@assert isfile(jaxcapse_python) jaxcapse_python
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = jaxcapse_python
ENV["JAX_ENABLE_X64"] = "true"
Pkg.instantiate()
println("PythonCall configured for the jaxcapse Poetry environment")

PythonCall configured for the jaxcapse Poetry environment


In [2]:
using Capse, PythonCall, Printf, Statistics

# [ln10As, ns, tau, H0, omega_b, omega_c, Mnu, w0, wa]; w0 + wa < -0.5.
params = [3.044, 0.965, 0.054, 67.4, 0.02237, 0.12, 0.06, -1.0, 0.0]
tt = Capse.trained_emulators["CAMB_MNUW0WACDM"]["TT"]
ell = Capse.get_ℓgrid(tt)
Dℓ_capse = Capse.get_Cℓ(params, tt)
@assert length(Dℓ_capse) == length(ell) == 9499

In [3]:
# Use the same Python model and CAMB reference worker as the artifact pipeline.
camb_root = get(ENV, "CAMB_COSMOREC_ROOT", joinpath(workspace, "tools", "CAMB-cosmorec"))
worker_root = joinpath(workspace, "emulator-zoo", "Capse.jl", "camb_mnuw0wacdm")
@assert isfile(joinpath(camb_root, "camb", "camblib.so")) camb_root
@assert isfile(joinpath(worker_root, "camb_worker.py")) worker_root
sys = pyimport("sys")
sys.path.insert(0, camb_root)
sys.path.insert(0, worker_root)
sys.path.insert(0, jaxcapse_root)
jax = pyimport("jax")
jax.config.update("jax_enable_x64", true)
jaxcapse = pyimport("jaxcapse")
worker = pyimport("camb_worker")
configuration = worker.backend_configuration()
@assert pyconvert(String, configuration["camb_version"]) == "2.0.4"
@assert pyconvert(String, configuration["recombination_model"]) == "CosmoRec"
println("Reference CAMB ", configuration["camb_version"], " / ", configuration["recombination_model"])

Reference CAMB 2.0.4 / CosmoRec


In [4]:
jnp = pyimport("jax.numpy")
np = pyimport("numpy")
python_params = jnp.array(pylist(params), dtype=jnp.float64)
jax_tt = jaxcapse.trained_emulators["camb_mnuw0wacdm"]["TT"]
Dℓ_jaxcapse = pyconvert(Vector{Float64}, np.asarray(jax_tt.get_Cl(python_params)))
names = ("ln10As", "ns", "tau", "H0", "omega_b", "omega_c", "Mnu", "w0", "wa")
camb_params = pydict(Dict(zip(names, params)))
camb_output = worker.compute_spectra(camb_params, 9500)
Dℓ_camb = pyconvert(Vector{Float64}, camb_output["TT_dense"])
@assert length(Dℓ_camb) == length(Dℓ_capse) == length(Dℓ_jaxcapse) == 9499
@assert all(isfinite, Dℓ_camb) && all(isfinite, Dℓ_capse) && all(isfinite, Dℓ_jaxcapse)

scale = maximum(abs, Dℓ_camb)
for (name, prediction) in (("Capse.jl", Dℓ_capse), ("jaxcapse", Dℓ_jaxcapse))
    residual = abs.(prediction .- Dℓ_camb) ./ scale
    @printf("%-10s median |ΔDℓ|/max|CAMB| = %.3e; max = %.3e\n", name, median(residual), maximum(residual))
end
@printf("At ℓ=200: CAMB %.8g, Capse.jl %.8g, jaxcapse %.8g\n", Dℓ_camb[199], Dℓ_capse[199], Dℓ_jaxcapse[199])
@printf("Max |Capse.jl - jaxcapse| = %.3e\n", maximum(abs.(Dℓ_capse .- Dℓ_jaxcapse)))

Capse.jl   median |ΔDℓ|/max|CAMB| = 6.328e-08; max = 1.613e-04
jaxcapse   median |ΔDℓ|/max|CAMB| = 6.328e-08; max = 1.613e-04
At ℓ=200: CAMB 5607.254, Capse.jl 5606.5362, jaxcapse 5606.5362
Max |Capse.jl - jaxcapse| = 1.046e-11
